# 07.1 — How LLMs Are Trained

The journey from a pile of web text to a chat assistant: pre-training, scaling laws, supervised fine-tuning, RLHF and DPO, reasoning models, and mixture of experts.
Each section starts with a one-line takeaway in bold, so you can review quickly.

## What's in this notebook

- 1\. The big picture: the training stages
- 2\. Pre-training data
- 3\. Scaling laws: how big, how much data
- 4\. Supervised fine-tuning (SFT)
- 5\. Preference tuning: RLHF
- 6\. Preference tuning: DPO and friends
- 7\. Reasoning models
- 8\. Mixture of experts (MoE)
- 9\. Base vs instruct, open vs closed
- 10\. Trade-offs
- 11\. Interview questions
- 12\. In my own words

In [1]:
import logging
import os

import numpy as np
import torch
import torch.nn.functional as F
from torch import nn

os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
from transformers import AutoTokenizer  # noqa: E402
from transformers.utils import logging as hf_logging  # noqa: E402

hf_logging.set_verbosity_error()
logging.getLogger("huggingface_hub").setLevel(logging.ERROR)
torch.manual_seed(0)
print("ready")

ready


# 1. THE BIG PICTURE: THE TRAINING STAGES

**In one line: pre-training teaches the model language and knowledge; post-training teaches it to be a helpful assistant.**

| Stage | Data | What the model learns | Cost |
|---|---|---|---|
| **pre-training** | trillions of tokens of web pages, books, code | predict the next token: grammar, facts, reasoning patterns | huge: thousands of GPUs for weeks or months |
| **mid-training** (optional) | high-quality and long documents, code, maths | longer context, better skills in chosen areas | moderate |
| **supervised fine-tuning (SFT)** | tens of thousands to millions of (prompt, good answer) examples | follow instructions, answer in a chat format | small |
| **preference tuning** (RLHF, DPO) | pairs of answers where humans or AI picked the better one | answer the way people prefer: helpful, honest, safe | small to moderate |
| **reinforcement learning for reasoning** | problems with checkable answers (maths, code) | think step by step before answering | moderate to large |

The model after pre-training is a **base model**: it continues text but doesn't follow instructions. After post-training it's an **instruct** (or chat) model.

# 2. PRE-TRAINING DATA

**In one line: data quality matters as much as model size; most of the work is filtering and cleaning.**

| Step | Why |
|---|---|
| collect | web crawls (Common Crawl), books, Wikipedia, code (GitHub), papers, maths |
| extract text | strip HTML, menus and ads; keep the main content |
| filter quality | drop spam, gibberish and machine-generated junk, often with a small classifier trained to recognise good pages |
| remove duplicates | duplicated text gets memorised and wastes compute; near-duplicates are found with MinHash |
| remove harmful content and personal data | safety and privacy |
| mix | choose the share of web, code, maths and languages; more code and maths improves reasoning |
| decontaminate | remove benchmark test questions, so evaluation scores stay honest |

Recent models train on **10–40 trillion tokens**. Running out of good public text is a real limit, which is why synthetic data (text written by other models) is now common.

# 3. SCALING LAWS: HOW BIG, HOW MUCH DATA

**In one line: loss falls predictably as a power law when you add parameters, data or compute; for a fixed budget, grow the model and the data together, about 20 tokens per parameter.**

- Training compute is about `C ≈ 6 × N × D` (N parameters, D training tokens), as in 06.3.
- **Kaplan et al. (2020)** found smooth power laws and suggested making models very large.
- **Chinchilla (Hoffmann et al., 2022)** fitted `loss = E + A / N^α + B / D^β` and showed earlier models were too big for their data.
  Their 70B model trained on 1.4T tokens beat much larger models.
- In practice, models now train far past "compute-optimal" (LLaMA 3 8B saw about 15T tokens, roughly 1,900 tokens per parameter):
  a smaller model trained longer costs more to train but much less to **serve**, and serving is where most money goes.

Below: the Chinchilla loss formula, with the paper's fitted constants, used to find the best model size for a given compute budget.

In [2]:
E, A, B, alpha, beta = 1.69, 406.4, 410.7, 0.34, 0.28        # fitted constants from the Chinchilla paper


def chinchilla_loss(N, D):
    return E + A / N ** alpha + B / D ** beta


print(f"{'compute (FLOPs)':>16} {'best model size':>16} {'training tokens':>16} {'tokens/param':>13} {'loss':>6}")
for C in [1e20, 1e21, 1e22, 1e23, 1e24]:
    N = np.logspace(7, 12, 4000)                    # try model sizes from 10M to 1T parameters
    D = C / (6 * N)                                  # the tokens this budget allows for each size
    best = np.argmin(chinchilla_loss(N, D))
    print(f"{C:>16.0e} {N[best] / 1e9:>14.2f}B {D[best] / 1e9:>14.0f}B {D[best] / N[best]:>13.0f} {chinchilla_loss(N[best], D[best]):>6.3f}")

 compute (FLOPs)  best model size  training tokens  tokens/param   loss
           1e+20           0.64B             26B            40  2.600
           1e+21           1.82B             91B            50  2.329
           1e+22           5.17B            323B            62  2.139
           1e+23          14.61B           1141B            78  2.005
           1e+24          41.30B           4036B            98  1.911


As the budget grows, the best model size and the data grow **together**: 10x more compute gives a model about 2.8x bigger, trained on about 3.5x more tokens.

This fitted formula suggests 40–100 tokens per parameter. The famous "about 20" comes from the paper's two other estimation methods, and a 2024 re-analysis found an error in this fit that, once corrected, agrees with about 20. For interviews, remember the shape: never spend the whole budget on model size alone.

# 4. SUPERVISED FINE-TUNING (SFT)

**In one line: show the base model many examples of good conversations, and train it with the same next-token loss, but only on the answer tokens.**

- Data: (instruction, ideal answer) pairs written by people or by stronger models, formatted with the model's **chat template** (06.1 section 6).
- The loss is ordinary next-token cross-entropy, but the prompt tokens are **masked** (label −100, ignored as in 02.2 section 8):
  the model should learn to *write answers*, not to predict the user's question.
- A few thousand very high-quality examples can matter more than a million mediocre ones ("quality over quantity").

Building the labels for one training example:

In [3]:
tok = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct")
messages = [{"role": "user", "content": "What is the capital of France?"},
            {"role": "assistant", "content": "The capital of France is Paris."}]

prompt_text = tok.apply_chat_template(messages[:1], tokenize=False, add_generation_prompt=True)   # everything before the answer
full_text = tok.apply_chat_template(messages, tokenize=False)
prompt_ids, full_ids = tok(prompt_text)["input_ids"], tok(full_text)["input_ids"]

labels = [-100] * len(prompt_ids) + full_ids[len(prompt_ids):]       # -100 = ignored by the loss
trained_on = tok.decode([t for t, l in zip(full_ids, labels) if l != -100])
print(f"{len(full_ids)} tokens in the example, {sum(l != -100 for l in labels)} of them trained on")
print("the loss only covers:", repr(trained_on))

45 tokens in the example, 9 of them trained on
the loss only covers: 'The capital of France is Paris.<|im_end|>\n'


Only the assistant's answer (and its end-of-message token) is trained on; the system prompt, the question and the formatting tokens are masked out.

# 5. PREFERENCE TUNING: RLHF

**In one line: train a reward model to predict which answer people prefer, then use reinforcement learning to make the LLM produce answers that score high, without drifting too far from the SFT model.**

The steps of **RLHF** (reinforcement learning from human feedback):
1. For each prompt, collect two or more answers and ask people which is better.
2. Train a **reward model** (an LLM with a single-number output) so the preferred answer scores higher.
   The loss is the **Bradley–Terry** loss: `−log σ(r_chosen − r_rejected)`.
3. Optimize the LLM with **PPO** to maximise the reward, minus a **KL penalty** that keeps it close to the SFT model.
   Without the penalty the model learns to "hack" the reward model with strange text that scores high.

Why prefer comparisons? Picking the better of two answers is much easier and more consistent for people than writing a perfect answer or giving a score from 1 to 10.

In [4]:
def reward_model_loss(r_chosen: torch.Tensor, r_rejected: torch.Tensor) -> torch.Tensor:
    return -F.logsigmoid(r_chosen - r_rejected).mean()          # Bradley-Terry: P(chosen wins) = σ(r_chosen - r_rejected)


for r_chosen, r_rejected in [(2.0, -1.0), (0.5, 0.5), (-1.0, 2.0)]:
    loss = reward_model_loss(torch.tensor([r_chosen]), torch.tensor([r_rejected]))
    print(f"reward chosen {r_chosen:>4}, rejected {r_rejected:>4}: P(chosen is better) = {torch.sigmoid(torch.tensor(r_chosen - r_rejected)):.2f}, loss {loss:.3f}")

reward chosen  2.0, rejected -1.0: P(chosen is better) = 0.95, loss 0.049
reward chosen  0.5, rejected  0.5: P(chosen is better) = 0.50, loss 0.693
reward chosen -1.0, rejected  2.0: P(chosen is better) = 0.05, loss 3.049


When the reward model already ranks the chosen answer higher, the loss is small; when it ranks them the wrong way round, the loss is large and training pushes the scores apart.

# 6. PREFERENCE TUNING: DPO AND FRIENDS

**In one line: DPO gets the same effect as RLHF with a simple classification-style loss on preference pairs: no reward model and no reinforcement learning loop.**

DPO (Direct Preference Optimization) compares how much the model being trained, relative to a frozen reference (the SFT model), raises the probability of the chosen answer versus the rejected one:

```text
margin = β · [ (log π(chosen) − log π_ref(chosen)) − (log π(rejected) − log π_ref(rejected)) ]
loss   = −log σ(margin)
```

- `log π(answer)` is the sum of the log-probabilities of the answer's tokens (picked with `gather`, 02.1 section 6).
- `β` controls how far the model may move from the reference: small β allows big changes.
- It's simpler, cheaper and more stable than PPO, which is why it's popular for open models. RLHF with PPO can still win at the largest scale.

In [5]:
def dpo_loss(policy_chosen, policy_rejected, ref_chosen, ref_rejected, beta=0.1):
    """Inputs are total log-probabilities of each answer under the trained model (policy) and the frozen reference."""
    margin = beta * ((policy_chosen - ref_chosen) - (policy_rejected - ref_rejected))
    return -F.logsigmoid(margin).mean(), margin


ref_chosen, ref_rejected = torch.tensor([-40.0]), torch.tensor([-38.0])          # the reference slightly prefers the rejected answer
for name, pc, pr in [("same as the reference", -40.0, -38.0),
                     ("chosen up 5, rejected down 5", -35.0, -43.0),
                     ("moved the wrong way", -45.0, -33.0)]:
    loss, margin = dpo_loss(torch.tensor([pc]), torch.tensor([pr]), ref_chosen, ref_rejected)
    print(f"{name:<30} margin {margin.item():+.2f}  loss {loss.item():.3f}")

same as the reference          margin +0.00  loss 0.693
chosen up 5, rejected down 5   margin +1.00  loss 0.313
moved the wrong way            margin -1.00  loss 1.313


At the start the model equals the reference (margin 0, loss ln 2 = 0.693). Raising the chosen answer's probability and lowering the rejected one lowers the loss; moving the wrong way raises it.

| Method | Idea |
|---|---|
| **RLHF + PPO** | reward model + reinforcement learning with a KL penalty |
| **DPO** | the preference loss above, no reward model |
| **KTO** | learns from single answers marked good or bad, no pairs needed |
| **ORPO / SimPO** | variants that fold preference learning into SFT or drop the reference model |
| **RLAIF / Constitutional AI** | an AI model, guided by written principles, produces the preference labels instead of people |

# 7. REASONING MODELS

**In one line: reasoning models are trained with reinforcement learning to write a long chain of thought before answering, and they get better the more "thinking" tokens they spend at test time.**

- **Chain-of-thought:** writing out intermediate steps improves answers on maths, logic and code, because each token gets more computation to build on.
- **Reinforcement learning with verifiable rewards:** give the model problems with checkable answers (a maths result, code that passes tests), let it try many times,
  and reward the attempts that are correct. Long, self-correcting reasoning emerges on its own.
- **GRPO** (used for DeepSeek-R1) scores each attempt against the average of a group of attempts for the same problem, so no separate value model is needed.
- **Test-time compute:** spending more tokens thinking (or sampling several answers and voting) trades latency and cost for accuracy.
- **Distillation:** the reasoning traces of a big model are used to fine-tune smaller models, which then reason surprisingly well.

| | Standard chat model | Reasoning model |
|---|---|---|
| answers | right away | after a (sometimes hidden) thinking phase |
| best at | conversation, writing, simple questions | maths, coding, planning, multi-step problems |
| cost and latency | lower | higher: many more output tokens |

# 8. MIXTURE OF EXPERTS (MOE)

**In one line: replace the feed-forward layer with many "expert" feed-forward layers and a router that sends each token to only a few of them, so the model has many parameters but uses only some per token.**

- A small **router** scores the experts for each token; the token goes to the **top-k** (often 2 of 8, or 8 of 256), and their outputs are mixed by the router's weights.
- **Total vs active parameters:** Mixtral 8x7B has about 47B parameters in total but uses about 13B per token; DeepSeek-V3 has 671B in total and about 37B active.
  Compute per token follows the *active* count; memory must hold the *total*.
- A **load-balancing loss** stops the router from sending everything to a few favourite experts.
- Downsides: all experts must sit in memory, and splitting experts across GPUs adds communication.

In [6]:
class MoELayer(nn.Module):
    def __init__(self, d: int = 64, n_experts: int = 8, top_k: int = 2):
        super().__init__()
        self.top_k = top_k
        self.router = nn.Linear(d, n_experts)
        self.experts = nn.ModuleList([nn.Sequential(nn.Linear(d, 4 * d), nn.GELU(), nn.Linear(4 * d, d)) for _ in range(n_experts)])

    def forward(self, x):                                       # x: (tokens, d)
        weights, chosen = self.router(x).topk(self.top_k, dim=-1)
        weights = weights.softmax(dim=-1)                       # mix only the chosen experts
        out = torch.zeros_like(x)
        for slot in range(self.top_k):
            for e, expert in enumerate(self.experts):
                hit = chosen[:, slot] == e                      # tokens sent to expert e in this slot
                if hit.any():
                    out[hit] += weights[hit, slot, None] * expert(x[hit])
        return out, chosen


moe = MoELayer()
tokens = torch.randn(1000, 64)
out, chosen = moe(tokens)
per_expert = sum(p.numel() for p in moe.experts[0].parameters())
print(f"output {tuple(out.shape)} | total expert parameters {8 * per_expert:,}, active per token {2 * per_expert:,} (2 of 8)")
print("tokens routed to each expert:", torch.bincount(chosen.flatten(), minlength=8).tolist())

output (1000, 64) | total expert parameters 264,704, active per token 66,176 (2 of 8)
tokens routed to each expert: [232, 252, 231, 254, 180, 257, 322, 272]


Each token used only 2 of the 8 experts, so a quarter of the expert parameters did the work for each token (66k of 265k). Even with an untrained router the load is uneven (180 to 322 tokens per expert). During training this imbalance tends to grow, which is why MoE training adds a load-balancing loss.

# 9. BASE VS INSTRUCT, OPEN VS CLOSED

**In one line: use an instruct model for applications; choose between open-weight models you host and closed models behind an API.**

| Choice | Options | Notes |
|---|---|---|
| model type | base / instruct (chat) / reasoning | base: continues text, used for further training. Instruct: follows instructions. Reasoning: thinks first |
| access | open-weight (LLaMA, Qwen, Mistral, Gemma, DeepSeek) / closed API (Claude, GPT, Gemini) | open: run anywhere, fine-tune freely, data stays with you, you run the servers. Closed: strongest models, no infrastructure, pay per token |
| size | ~1–8B / ~30–70B / 100B+ and MoE | small: cheap, laptop or single GPU. Large: better quality, need several GPUs |
| licence | Apache 2.0 / MIT / custom licences | check commercial-use limits before shipping |

# 10. TRADE-OFFS

| Decision | Options | What you gain and what you pay | Usual choice |
|---|---|---|---|
| Model size vs training tokens | compute-optimal (~20 tokens/param) / train much longer | compute-optimal: best loss per training dollar. Longer: a smaller model that is cheaper to serve | train longer when the model will serve lots of traffic |
| Alignment method | RLHF + PPO / DPO | PPO: flexible, can reach the best results, complex and unstable. DPO: simple and stable, offline | DPO for most teams, PPO-style RL at frontier labs |
| Feedback source | human / AI (RLAIF) | human: trusted, slow and expensive. AI: cheap and scalable, can copy the judge's biases | a mix: AI labels checked by people |
| Dense vs MoE | dense / mixture of experts | dense: simple, less memory. MoE: more knowledge per unit of compute, all experts must fit in memory | MoE for the largest models |
| Reasoning | standard model / reasoning model | reasoning: much better at hard multi-step problems, slower and pricier | reasoning models for maths, code and planning; standard for chat and simple tasks |
| Data | more data / cleaner data | cleaner data often beats more data | heavy filtering and deduplication |

# 11. INTERVIEW QUESTIONS

### Q1. How is an LLM like ChatGPT or Claude trained, in 30 seconds?
Pre-training on trillions of tokens with next-token prediction gives a base model with broad knowledge. Supervised fine-tuning on example conversations teaches it to follow instructions.
Preference tuning (RLHF or DPO) makes its answers more helpful and safe, and reinforcement learning on checkable problems teaches step-by-step reasoning.

### Q2. What are scaling laws, and what did Chinchilla show?
Loss falls as a smooth power law in parameters, data and compute. Chinchilla showed that for a fixed compute budget, parameters and tokens should grow together, about 20 tokens per parameter,
so many earlier models were too large and undertrained.

### Q3. Why do modern models train on far more than 20 tokens per parameter?
Chinchilla optimizes training cost only. A smaller model trained on more data reaches similar quality and is much cheaper and faster to run for millions of users, so the extra training pays off at inference.

### Q4. What is SFT, and why mask the prompt tokens?
Supervised fine-tuning trains on (prompt, ideal answer) conversations with the next-token loss. The prompt tokens get label −100 so only the answer is learned:
the goal is to generate good responses, not to model user questions.

### Q5. Explain RLHF.
Collect human comparisons between answers, train a reward model with the Bradley–Terry loss to score preferred answers higher, then optimize the LLM with PPO to maximize reward
minus a KL penalty that keeps it close to the SFT model and prevents reward hacking.

### Q6. What is DPO, and why is it popular?
Direct Preference Optimization turns the RLHF objective into a simple loss on preference pairs: increase the chosen answer's log-probability relative to a frozen reference more than the rejected one's.
No reward model and no RL loop, so it's simpler, cheaper and more stable.

### Q7. What is reward hacking?
The policy finds outputs that score high with the reward model without actually being better, such as overly long or flattering answers, because the reward model is an imperfect proxy.
KL penalties, better reward models and human checks reduce it.

### Q8. What makes reasoning models different?
They're trained with reinforcement learning on verifiable tasks to produce a long chain of thought before the final answer, and their accuracy improves with more test-time computation.
They're stronger at maths, code and planning, but slower and more expensive per answer.

### Q9. What is a mixture of experts, and what are total vs active parameters?
The feed-forward layer is replaced by many experts and a router that sends each token to the top-k of them. Total parameters (all experts) set the memory needed;
active parameters (the experts used per token) set the compute. A load-balancing loss keeps the experts evenly used.

### Q10. Base model vs instruct model?
A base model only continues text in the style of its training data. An instruct model has been post-trained (SFT and preference tuning) to follow instructions in a chat format;
it's what applications use. Base models are the starting point for your own fine-tuning.

### Q11. Why does data deduplication matter in pre-training?
Duplicated text is memorized and repeated, wastes compute, and can leak benchmark answers. Removing exact and near duplicates (MinHash) improves quality and makes evaluation honest.

### Q12. What is RLAIF or Constitutional AI?
Using an AI model, guided by a written set of principles, to produce the preference labels or critiques instead of humans. It scales feedback cheaply; humans still check the results.

# 12. IN MY OWN WORDS

Write this from memory, without looking above:

- The stages from raw text to a chat assistant:
- What Chinchilla says, and why models train longer anyway:
- SFT, and why prompt tokens are masked:
- RLHF vs DPO:
- What MoE and reasoning models change: